# Meta-Learner Comparison for Stacking Ensemble
## Energy Theft Detection — TDD2022 Dataset

---

### Purpose of this notebook
After building the stacking ensemble (RF + XGB + MLP as base models), we need to **select the best meta-learner** in a methodologically sound way. This notebook:

1. Generates **Out-of-Fold (OOF) meta-features** from the base models on training data only
2. Trains and evaluates **6 candidate meta-learners** using cross-validation (no test set leakage)
3. Compares them across **5 criteria**: CV Accuracy, Calibration (ECE), Training Time, Interpretability, Stability
4. Runs **Friedman + Nemenyi statistical tests** to identify which differences are real
5. Produces a **final scorecard table** for thesis use
6. Evaluates the winning meta-learner on the **test set** (done only once, at the very end)

### Scenarios covered: P7C and P6C (Known Consumer)

---

### Key methodological rule followed:
> Meta-learner selection is performed **exclusively on training data via cross-validation**. The test set is only used once, for the final evaluation of the selected model. This prevents test set leakage and ensures honest reporting.

---
## CELL 1 — Install Required Libraries
Run this cell first if any packages are missing in your environment.

In [ ]:
# Uncomment and run if packages are not already installed
# !pip install scikit-learn xgboost scikit-posthocs pandas numpy matplotlib seaborn scipy joblib

print('If no errors above, all packages are already installed.')
print('Proceed to Cell 2.')

---
## CELL 2 — Import All Libraries
Imports everything needed across the entire notebook up front.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import time
import warnings
warnings.filterwarnings('ignore')

# ── Data ──────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Sklearn: preprocessing ────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, cross_validate
)

# ── Sklearn: base models ──────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# ── Sklearn: meta-learner candidates ─────────────────────────────────────────
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier

# ── Sklearn: metrics ──────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# ── Statistical tests ─────────────────────────────────────────────────────────
from scipy.stats import friedmanchisquare
try:
    import scikit_posthocs as sp
    POSTHOCS_AVAILABLE = True
    print('scikit-posthocs available — Nemenyi test will run.')
except ImportError:
    POSTHOCS_AVAILABLE = False
    print('scikit-posthocs not found. Install with: pip install scikit-posthocs')
    print('Friedman test will still run; Nemenyi post-hoc will be skipped.')

# ── Global settings ───────────────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.20
N_CV_FOLDS   = 5

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('\nAll imports successful!')
print(f'Random state : {RANDOM_STATE}')
print(f'Test size    : {TEST_SIZE*100:.0f}%')
print(f'CV folds     : {N_CV_FOLDS}')

---
## CELL 3 — Load Dataset
Load the TDD2022 dataset. Update `DATASET_PATH`, `TARGET_COL`, and `CONSUMER_TYPE_COL` to match your file.

In [ ]:
# ── CONFIGURE THESE ───────────────────────────────────────────────────────────
DATASET_PATH      = '../data/Dataset_actual.csv'   # <-- UPDATE: path to your dataset
import os; os.makedirs('../results', exist_ok=True)
TARGET_COL = 'theft'              # <-- UPDATE: name of your target column
CONSUMER_TYPE_COL = 'Class'      # <-- UPDATE: name of consumer type column
THEFT6_LABEL_NAME = 'Theft6'             # <-- UPDATE: exact string name of Theft6 class
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATASET_PATH)

print(f'Dataset shape : {df.shape}')
print(f'Columns       : {df.columns.tolist()}')
print(f'\nTarget distribution:')
print(df[TARGET_COL].value_counts())
print(f'\nMissing values: {df.isnull().sum().sum()} total')

---
## CELL 4 — Preprocessing
Mirrors the paper exactly:
- Drop nulls
- Label-encode the consumer_type column (category codes)
- Label-encode the target
- Standard scaling is applied later (after splitting) to prevent data leakage

In [ ]:
# Drop missing values
df.dropna(inplace=True)
print(f'Shape after dropping NaNs: {df.shape}')

# Encode consumer type (categorical → integer codes)
df[CONSUMER_TYPE_COL] = df[CONSUMER_TYPE_COL].astype('category').cat.codes

# Encode target labels
le = LabelEncoder()
df[TARGET_COL] = le.fit_transform(df[TARGET_COL])

# Identify feature columns
all_feature_cols  = [c for c in df.columns if c != TARGET_COL]   # 11 features (incl. consumer type)
consumption_cols  = [c for c in all_feature_cols if c != CONSUMER_TYPE_COL]  # 10 consumption only

# Get encoded integer for Theft6 (needed to build P6C)
theft6_encoded = le.transform([THEFT6_LABEL_NAME])[0]

print(f'\nAll feature cols  ({len(all_feature_cols)}): {all_feature_cols}')
print(f'Consumption cols  ({len(consumption_cols)}): {consumption_cols}')
print(f'Target classes after encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'Theft6 encoded as: {theft6_encoded}')

---
## CELL 5 — Build P7C and P6C Scenario Data
- **P7C**: 7 classes (Normal + Theft1–6), 11 features (includes consumer type)
- **P6C**: 6 classes (Normal + Theft1–5, Theft6 excluded), 11 features

In [ ]:
# ── P7C ───────────────────────────────────────────────────────────────────────
X_p7c = df[all_feature_cols].values
y_p7c = df[TARGET_COL].values
class_names_p7c = [str(c) for c in le.classes_]
n_classes_p7c   = len(class_names_p7c)

# ── P6C ───────────────────────────────────────────────────────────────────────
df_p6c = df[df[TARGET_COL] != theft6_encoded].copy()
X_p6c  = df_p6c[all_feature_cols].values
y_p6c_raw = df_p6c[TARGET_COL].values

# Re-encode P6C labels to be contiguous (0 to 5)
le_p6c = LabelEncoder()
y_p6c = le_p6c.fit_transform(y_p6c_raw)
class_names_p6c = [str(c) for c in le_p6c.classes_]
n_classes_p6c   = len(class_names_p6c)

print('=== P7C ===')
print(f'  X shape   : {X_p7c.shape}')
print(f'  Classes   : {class_names_p7c}')
print()
print('=== P6C ===')
print(f'  X shape   : {X_p6c.shape}')
print(f'  Classes   : {class_names_p6c}')

---
## CELL 6 — Train/Test Split and Standard Scaling
80% train / 20% test. Scaler is **fit on train only** to prevent leakage.

In [ ]:
def split_and_scale(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """Stratified split + StandardScaler fit on train only."""
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)
    return X_tr, X_te, y_tr, y_te, scaler

X_train_p7c, X_test_p7c, y_train_p7c, y_test_p7c, scaler_p7c = split_and_scale(X_p7c, y_p7c)
X_train_p6c, X_test_p6c, y_train_p6c, y_test_p6c, scaler_p6c = split_and_scale(X_p6c, y_p6c)

print('=== P7C Split ===')
print(f'  Train: {X_train_p7c.shape}  |  Test: {X_test_p7c.shape}')
print()
print('=== P6C Split ===')
print(f'  Train: {X_train_p6c.shape}  |  Test: {X_test_p6c.shape}')

---
## CELL 7 — Define Base Models
Same three base models as the original paper, with paper-matching hyperparameters.
These are fixed and do **not** change across meta-learner experiments.

In [ ]:
def get_base_models():
    """
    Returns the three base models from the paper.
    RF  : Random Forest   — 100 trees, Gini
    XGB : XGBoost         — max_depth=6, eta=0.3
    MLP : Multi-layer Perceptron — hidden=(100,), ReLU, Adam
    """
    rf = RandomForestClassifier(
        n_estimators=100, criterion='gini',
        random_state=0, min_samples_split=2, min_samples_leaf=1
    )
    xgb = XGBClassifier(
        max_depth=6, learning_rate=0.3, scale_pos_weight=1,
        min_child_weight=1, booster='gbtree',
        use_label_encoder=False, eval_metric='mlogloss',
        random_state=RANDOM_STATE, verbosity=0
    )
    mlp = MLPClassifier(
        hidden_layer_sizes=(100,), activation='relu',
        solver='adam', alpha=0.0001,
        max_iter=200, random_state=RANDOM_STATE
    )
    return [('RF', rf), ('XGB', xgb), ('MLP', mlp)]

print('Base models defined:')
for name, _ in get_base_models():
    print(f'  • {name}')
print('\nThese are fixed across ALL meta-learner experiments.')

---
## CELL 8 — Define All Meta-Learner Candidates

Six meta-learners are compared across the two scenarios:

| # | Meta-Learner | Why included |
|---|---|---|
| 1 | Logistic Regression | Baseline; interpretable; well-calibrated |
| 2 | Ridge Classifier | Regularized linear; fast; less sensitive to multicollinearity |
| 3 | Decision Tree | Interpretable; non-linear boundaries |
| 4 | SVM (RBF kernel) | Strong non-linear separator; good with probability features |
| 5 | Gradient Boosting | Sequential learner; can exploit patterns in meta-features |
| 6 | Extra Trees | Fast ensemble; high variance base models stabilised by averaging |

In [ ]:
def get_meta_learner_candidates():
    """
    Returns dict of {name: (model, is_interpretable)}.
    is_interpretable = True if model provides coefficients or feature importances.
    """
    candidates = {
        'Logistic Regression': (
            LogisticRegression(
                max_iter=1000, solver='lbfgs',
                multi_class='multinomial', C=1.0,
                random_state=RANDOM_STATE
            ),
            True   # interpretable via coefficients
        ),
        'Ridge Classifier': (
            CalibratedClassifierCV(
                RidgeClassifier(alpha=1.0, random_state=RANDOM_STATE),
                method='isotonic', cv=3
            ),
            True   # interpretable via wrapped coef_
        ),
        'Decision Tree': (
            DecisionTreeClassifier(
                max_depth=5, random_state=RANDOM_STATE
            ),
            True   # interpretable via feature importance + tree structure
        ),
        'SVM (RBF)': (
            SVC(
                kernel='rbf', C=1.0, gamma='scale',
                probability=True, random_state=RANDOM_STATE
            ),
            False  # black box
        ),
        'Gradient Boosting': (
            GradientBoostingClassifier(
                n_estimators=100, learning_rate=0.1,
                max_depth=3, random_state=RANDOM_STATE
            ),
            False  # black box (feature importance only)
        ),
        'Extra Trees': (
            ExtraTreesClassifier(
                n_estimators=100, random_state=RANDOM_STATE
            ),
            False  # black box (feature importance only)
        ),
    }
    return candidates

print('Meta-learner candidates:')
for i, (name, (_, interp)) in enumerate(get_meta_learner_candidates().items(), 1):
    interp_str = '✓ Interpretable' if interp else '✗ Black box'
    print(f'  {i}. {name:<25} {interp_str}')

---
## CELL 9 — Generate Out-of-Fold (OOF) Meta-Features

**What this does and why it matters:**

Each base model is trained on k-1 folds and predicts on the held-out fold. This produces **out-of-fold probability predictions** — unbiased estimates of how the base model would perform on unseen data. These OOF predictions become the input features for the meta-learner.

This is done on **training data only**. The test set is not touched here.

OOF matrix shape: `(n_train_samples, n_base_models × n_classes)`
- P7C: `(n_train, 3 × 7)` = `(n_train, 21)`
- P6C: `(n_train, 3 × 6)` = `(n_train, 18)`

In [ ]:
def generate_oof_meta_features(X_train, y_train, n_classes, n_folds=N_CV_FOLDS):
    """
    Generates Out-of-Fold probability predictions from all base models.
    Also trains each base model on the full training set for later
    generation of test-set meta-features.

    Returns:
        oof_matrix      : (n_train, n_base_models * n_classes) array
        trained_bases   : list of fully-trained base models (on full train set)
    """
    base_models    = get_base_models()
    n_base         = len(base_models)
    n_train        = X_train.shape[0]
    oof_matrix     = np.zeros((n_train, n_base * n_classes))
    skf            = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    print(f'  Generating OOF meta-features using {n_folds}-fold CV...')

    for b_idx, (name, model) in enumerate(base_models):
        print(f'    Base model {b_idx+1}/{n_base}: {name}')
        col_start = b_idx * n_classes
        col_end   = col_start + n_classes

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            X_tr_fold, X_val_fold = X_train[tr_idx], X_train[val_idx]
            y_tr_fold             = y_train[tr_idx]
            # Clone model to avoid cross-fold contamination
            from sklearn.base import clone
            fold_model = clone(model)
            fold_model.fit(X_tr_fold, y_tr_fold)
            oof_matrix[val_idx, col_start:col_end] = fold_model.predict_proba(X_val_fold)

    # Train each base model on FULL training set (for test-set meta-features later)
    trained_bases = []
    print('  Training base models on full training set...')
    for name, model in base_models:
        model.fit(X_train, y_train)
        trained_bases.append((name, model))

    print(f'  OOF matrix shape: {oof_matrix.shape}')
    return oof_matrix, trained_bases


def generate_test_meta_features(X_test, trained_bases, n_classes):
    """
    Uses fully-trained base models to generate meta-features for the test set.
    Only called at the very end for final evaluation.
    """
    n_base       = len(trained_bases)
    test_matrix  = np.zeros((X_test.shape[0], n_base * n_classes))
    for b_idx, (name, model) in enumerate(trained_bases):
        col_start = b_idx * n_classes
        col_end   = col_start + n_classes
        test_matrix[:, col_start:col_end] = model.predict_proba(X_test)
    return test_matrix


# ── Run OOF generation for both scenarios ────────────────────────────────────
print('Generating OOF meta-features for P7C...')
oof_p7c, trained_bases_p7c = generate_oof_meta_features(X_train_p7c, y_train_p7c, n_classes_p7c)

print('\nGenerating OOF meta-features for P6C...')
oof_p6c, trained_bases_p6c = generate_oof_meta_features(X_train_p6c, y_train_p6c, n_classes_p6c)

print('\nOOF meta-feature generation complete.')
print(f'  P7C OOF shape: {oof_p7c.shape}  (= n_train × [3 base models × {n_classes_p7c} classes])')
print(f'  P6C OOF shape: {oof_p6c.shape}  (= n_train × [3 base models × {n_classes_p6c} classes])')

---
## CELL 10 — Define ECE (Expected Calibration Error) Calculator

**What ECE measures:**
Calibration is the agreement between a model's predicted confidence and actual accuracy. If a model says "80% confident" for 100 predictions, ~80 should be correct.

ECE = weighted average of |predicted confidence − actual accuracy| across probability bins.
**Lower ECE = better calibrated.** For a decision support system, calibration is critical — operators act on confidence scores.

In [ ]:
def compute_ece(y_true, y_prob, n_bins=10):
    """
    Computes Expected Calibration Error (ECE) for multi-class predictions.
    Uses the confidence of the predicted class (max probability).

    ECE = Σ (|bin| / N) × |accuracy(bin) − confidence(bin)|

    Args:
        y_true : true class labels (1D array)
        y_prob : predicted probabilities (2D array, shape n_samples × n_classes)
        n_bins : number of calibration bins (default 10)

    Returns:
        ece (float) — lower is better
    """
    confidences = np.max(y_prob, axis=1)       # confidence = max probability
    predictions = np.argmax(y_prob, axis=1)    # predicted class
    correct     = (predictions == y_true).astype(float)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n   = len(y_true)

    for i in range(n_bins):
        in_bin = (confidences > bin_edges[i]) & (confidences <= bin_edges[i + 1])
        if in_bin.sum() == 0:
            continue
        bin_acc  = correct[in_bin].mean()
        bin_conf = confidences[in_bin].mean()
        ece     += (in_bin.sum() / n) * abs(bin_acc - bin_conf)

    return ece


print('ECE function defined.')
print('Lower ECE = better calibration = more trustworthy confidence scores.')

---
## CELL 11 — Run Meta-Learner Comparison (CV on OOF Features)

For each meta-learner candidate, using only the OOF training matrix:
- **CV Accuracy**: 5-fold stratified CV mean ± std
- **CV F1 (macro)**: 5-fold macro F1 mean
- **Stability**: std of CV accuracy across folds (lower = more stable)
- **Training time**: time to fit on full OOF training matrix
- **ECE**: calibration error on OOF held-out predictions
- **Interpretable**: whether the model exposes coefficients/feature importance

⚠️ **No test set is used in this cell.**

In [ ]:
def compare_meta_learners(oof_X, y_train, scenario_name, n_folds=N_CV_FOLDS):
    """
    Evaluates all meta-learner candidates on OOF meta-features using CV.
    Returns a results dict and a per-fold accuracy matrix (for statistical tests).
    """
    candidates       = get_meta_learner_candidates()
    skf              = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    results          = {}
    fold_scores_all  = {}   # {meta_name: [fold1_acc, fold2_acc, ...]}

    print(f'\n{"="*60}')
    print(f'  META-LEARNER COMPARISON — {scenario_name}')
    print(f'{"="*60}')

    for meta_name, (meta_model, is_interpretable) in candidates.items():
        print(f'  Evaluating: {meta_name}...')
        from sklearn.base import clone

        fold_accs  = []
        fold_f1s   = []
        ece_scores = []

        for tr_idx, val_idx in skf.split(oof_X, y_train):
            X_tr, X_val = oof_X[tr_idx], oof_X[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            m = clone(meta_model)
            m.fit(X_tr, y_tr)

            y_pred = m.predict(X_val)
            y_prob = m.predict_proba(X_val)

            fold_accs.append(accuracy_score(y_val, y_pred))
            fold_f1s.append(f1_score(y_val, y_pred, average='macro', zero_division=0))
            ece_scores.append(compute_ece(y_val, y_prob))

        # Training time on full OOF matrix
        m_full = clone(meta_model)
        t0     = time.time()
        m_full.fit(oof_X, y_train)
        train_time = time.time() - t0

        results[meta_name] = {
            'CV Accuracy Mean':  np.mean(fold_accs) * 100,
            'CV Accuracy Std':   np.std(fold_accs)  * 100,
            'CV F1 Macro Mean':  np.mean(fold_f1s)  * 100,
            'ECE (↓ better)':    np.mean(ece_scores),
            'Train Time (s)':    train_time,
            'Interpretable':     '✓' if is_interpretable else '✗',
            'Stability (std↓)':  np.std(fold_accs) * 100,
            'trained_model':     m_full
        }
        fold_scores_all[meta_name] = fold_accs

        print(f'    CV Acc: {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%  '
              f'| F1: {np.mean(fold_f1s)*100:.2f}%  '
              f'| ECE: {np.mean(ece_scores):.4f}  '
              f'| Time: {train_time:.2f}s')

    return results, fold_scores_all


# ── Run comparison for both scenarios ────────────────────────────────────────
results_p7c, fold_scores_p7c = compare_meta_learners(oof_p7c, y_train_p7c, 'P7C (7 classes, Known Consumer)')
results_p6c, fold_scores_p6c = compare_meta_learners(oof_p6c, y_train_p6c, 'P6C (6 classes, Known Consumer)')

---
## CELL 12 — Friedman Statistical Test

**Why Friedman and not ANOVA?**
ANOVA assumes normality and equal variance — not guaranteed with classifier accuracies. The Friedman test is non-parametric and designed exactly for comparing multiple classifiers across multiple folds.

- **H₀**: All meta-learners perform equally (no significant difference)
- **H₁**: At least one meta-learner differs significantly
- If **p < 0.05**, reject H₀ → differences are real → run Nemenyi post-hoc

In [ ]:
def run_friedman_test(fold_scores_dict, scenario_name):
    """
    Runs Friedman test across all meta-learners' per-fold CV accuracy scores.
    fold_scores_dict: {meta_name: [fold1_acc, fold2_acc, ...]}
    """
    names  = list(fold_scores_dict.keys())
    scores = [fold_scores_dict[n] for n in names]

    stat, p = friedmanchisquare(*scores)

    print(f'\n{"─"*55}')
    print(f'  Friedman Test — {scenario_name}')
    print(f'{"─"*55}')
    print(f'  Chi-square statistic : {stat:.4f}')
    print(f'  p-value              : {p:.6f}')

    if p < 0.05:
        print(f'  Result  : ✓ SIGNIFICANT (p < 0.05)')
        print(f'  Meaning : At least one meta-learner is statistically different.')
        print(f'  Action  : Proceed to Nemenyi post-hoc test to identify which pairs differ.')
    else:
        print(f'  Result  : ✗ NOT significant (p ≥ 0.05)')
        print(f'  Meaning : No statistically significant difference between meta-learners.')
        print(f'  Action  : Select based on secondary criteria (ECE, speed, interpretability).')

    return p, names, scores


p_p7c, names_p7c, scores_p7c = run_friedman_test(fold_scores_p7c, 'P7C')
p_p6c, names_p6c, scores_p6c = run_friedman_test(fold_scores_p6c, 'P6C')

---
## CELL 13 — Nemenyi Post-Hoc Test (if Friedman is significant)

The Friedman test tells us *some* difference exists but not *which* pairs differ. The **Nemenyi test** does pairwise comparisons with multiple-comparison correction.

- Values in the matrix are **p-values** for each pair of meta-learners
- **p < 0.05** → the two meta-learners are statistically different
- **p ≥ 0.05** → the two are statistically indistinguishable (use other criteria to choose)

In [ ]:
def run_nemenyi_test(fold_scores_dict, p_friedman, scenario_name):
    """
    Runs Nemenyi post-hoc test if Friedman test was significant.
    Plots a heatmap of pairwise p-values.
    """
    if not POSTHOCS_AVAILABLE:
        print('scikit-posthocs not installed — skipping Nemenyi test.')
        print('Install with: pip install scikit-posthocs')
        return None

    if p_friedman >= 0.05:
        print(f'Friedman test was not significant for {scenario_name}.')
        print('Nemenyi test not applicable — differences are not statistically real.')
        return None

    names  = list(fold_scores_dict.keys())
    data   = np.array([fold_scores_dict[n] for n in names]).T  # shape: (n_folds, n_models)
    df_nemenyi = pd.DataFrame(data, columns=names)

    nemenyi_p = sp.posthoc_nemenyi_friedman(df_nemenyi)

    print(f'\nNemenyi Post-Hoc Test — {scenario_name}')
    print('p-value matrix (< 0.05 = significantly different pair):')

    # Plot heatmap
    fig, ax = plt.subplots(figsize=(9, 7))
    mask = np.eye(len(names), dtype=bool)  # mask diagonal
    sns.heatmap(
        nemenyi_p, annot=True, fmt='.3f', cmap='RdYlGn',
        vmin=0, vmax=0.1, center=0.05,
        xticklabels=names, yticklabels=names,
        linewidths=0.5, ax=ax
    )
    ax.set_title(
        f'Nemenyi Post-Hoc p-values — {scenario_name}\n'
        f'(Green = significantly different | Red = not significantly different)',
        fontsize=11
    )
    plt.xticks(rotation=30, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(f'../results/nemenyi_{scenario_name.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return nemenyi_p


nemenyi_p7c = run_nemenyi_test(fold_scores_p7c, p_p7c, 'P7C')
nemenyi_p6c = run_nemenyi_test(fold_scores_p6c, p_p6c, 'P6C')

---
## CELL 14 — Calibration Reliability Diagrams

A reliability diagram plots **mean predicted probability vs actual fraction of positives** per bin. A perfectly calibrated model follows the diagonal (y = x).

This is plotted for each meta-learner on its OOF held-out predictions, for each scenario.

In [ ]:
def plot_calibration_curves(oof_X, y_train, results_dict, scenario_name, n_bins=10):
    """
    Plots reliability diagrams for all meta-learners.
    Uses OOF data to generate probabilities for the positive class (macro).
    """
    candidates = get_meta_learner_candidates()
    n_models   = len(candidates)
    ncols      = 3
    nrows      = (n_models + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
    axes = axes.flatten()

    skf = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    for ax_idx, (meta_name, (meta_model, _)) in enumerate(candidates.items()):
        from sklearn.base import clone
        all_conf  = []
        all_corr  = []

        for tr_idx, val_idx in skf.split(oof_X, y_train):
            X_tr, X_val = oof_X[tr_idx], oof_X[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]
            m = clone(meta_model)
            m.fit(X_tr, y_tr)
            probs = m.predict_proba(X_val)
            preds = np.argmax(probs, axis=1)
            all_conf.extend(np.max(probs, axis=1))
            all_corr.extend((preds == y_val).astype(float))

        all_conf = np.array(all_conf)
        all_corr = np.array(all_corr)

        # Bin and compute
        bin_edges   = np.linspace(0, 1, n_bins + 1)
        bin_centers = []
        bin_accs    = []
        for i in range(n_bins):
            mask = (all_conf > bin_edges[i]) & (all_conf <= bin_edges[i+1])
            if mask.sum() > 0:
                bin_centers.append(all_conf[mask].mean())
                bin_accs.append(all_corr[mask].mean())

        ece = results_dict[meta_name]['ECE (↓ better)']
        ax  = axes[ax_idx]
        ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
        ax.plot(bin_centers, bin_accs, 'o-', color='steelblue', lw=2, label='Model')
        ax.fill_between(bin_centers, bin_accs, bin_centers,
                        alpha=0.15, color='tomato', label='Calibration gap')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(f'{meta_name}\nECE = {ece:.4f}', fontsize=10)
        ax.set_xlabel('Mean predicted probability')
        ax.set_ylabel('Fraction correct')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    # Hide unused axes
    for i in range(ax_idx + 1, len(axes)):
        axes[i].set_visible(False)

    fig.suptitle(f'Reliability Diagrams (Calibration) — {scenario_name}',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'../results/calibration_{scenario_name.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()


print('Plotting calibration curves for P7C...')
plot_calibration_curves(oof_p7c, y_train_p7c, results_p7c, 'P7C')

print('\nPlotting calibration curves for P6C...')
plot_calibration_curves(oof_p6c, y_train_p6c, results_p6c, 'P6C')

---
## CELL 15 — CV Accuracy Distribution (Box Plots)

Box plots show the distribution of per-fold accuracy for each meta-learner. A tighter box = more stable across folds. Outlier folds (dots outside whiskers) indicate instability.

In [ ]:
def plot_cv_boxplots(fold_scores_p7c, fold_scores_p6c):
    """
    Side-by-side box plots of per-fold CV accuracy for all meta-learners.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

    for ax, fold_scores, scenario in [
        (axes[0], fold_scores_p7c, 'P7C (7 classes, Known Consumer)'),
        (axes[1], fold_scores_p6c, 'P6C (6 classes, Known Consumer)')
    ]:
        names  = list(fold_scores.keys())
        data   = [np.array(fold_scores[n]) * 100 for n in names]

        bp = ax.boxplot(
            data, patch_artist=True, notch=False,
            medianprops={'color': 'black', 'linewidth': 2},
            whiskerprops={'linewidth': 1.5},
            capprops={'linewidth': 1.5},
            flierprops={'marker': 'o', 'markersize': 5, 'alpha': 0.5}
        )

        colors = plt.cm.Set2.colors
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.75)

        ax.set_xticks(range(1, len(names) + 1))
        ax.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=8)
        ax.set_title(f'CV Accuracy Distribution\n{scenario}', fontsize=11)
        ax.set_ylabel('CV Accuracy (%)')
        ax.set_xlabel('Meta-Learner')
        ax.grid(True, axis='y', alpha=0.4)

        # Annotate medians
        for i, d in enumerate(data):
            ax.text(i + 1, np.median(d) + 0.05, f'{np.median(d):.2f}%',
                    ha='center', va='bottom', fontsize=7, fontweight='bold')

    plt.suptitle('Per-Fold CV Accuracy — All Meta-Learner Candidates', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/cv_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_cv_boxplots(fold_scores_p7c, fold_scores_p6c)

---
## CELL 16 — Build the Final Scorecard Table

This table combines all five evaluation criteria into one structured comparison for **thesis use**.

The criteria and their direction:
- **CV Accuracy** ↑ higher is better
- **ECE** ↓ lower is better (calibration)
- **Training Time** ↓ lower is better
- **Interpretable** — binary (✓/✗)
- **Stability (std)** ↓ lower is better

In [ ]:
def build_scorecard(results_dict, scenario_name):
    """
    Builds a clean scorecard DataFrame from results dict.
    Highlights best value per column.
    """
    rows = []
    for meta_name, vals in results_dict.items():
        rows.append({
            'Meta-Learner':        meta_name,
            'CV Accuracy (%) ↑':   round(vals['CV Accuracy Mean'], 2),
            'CV F1 Macro (%) ↑':   round(vals['CV F1 Macro Mean'], 2),
            'ECE ↓':               round(vals['ECE (↓ better)'], 4),
            'Train Time (s) ↓':    round(vals['Train Time (s)'], 3),
            'Stability Std (%) ↓': round(vals['Stability (std↓)'], 4),
            'Interpretable':       vals['Interpretable'],
        })

    df_score = pd.DataFrame(rows).set_index('Meta-Learner')

    print(f'\n{"="*70}')
    print(f'  SCORECARD — {scenario_name}')
    print(f'{"="*70}')

    # Style for display
    styled = df_score.style \\n
        .highlight_max(subset=['CV Accuracy (%) ↑', 'CV F1 Macro (%) ↑'],
                       color='#c6efce') \
        .highlight_min(subset=['ECE ↓', 'Train Time (s) ↓', 'Stability Std (%) ↓'],
                       color='#c6efce') \
        .set_caption(f'Meta-Learner Scorecard — {scenario_name}') \
        .set_properties(**{'text-align': 'center'})

    display(styled)
    return df_score


scorecard_p7c = build_scorecard(results_p7c, 'P7C (7 classes, Known Consumer)')
scorecard_p6c = build_scorecard(results_p6c, 'P6C (6 classes, Known Consumer)')

---
## CELL 17 — Scorecard Radar (Spider) Chart

A radar chart gives a visual at-a-glance comparison across all 5 criteria simultaneously. Each axis is normalised 0–1 so that **outward always means better**.

In [ ]:
def plot_radar_chart(scorecard_df, scenario_name):
    """
    Radar chart comparing meta-learners across all scorecard criteria.
    All axes normalised so outward = better.
    """
    # Numeric columns only
    numeric_cols = ['CV Accuracy (%) ↑', 'CV F1 Macro (%) ↑',
                    'ECE ↓', 'Train Time (s) ↓', 'Stability Std (%) ↓']
    df_num = scorecard_df[numeric_cols].copy()

    # Normalise: for ↑ columns, higher=better; for ↓ columns, invert
    df_norm = df_num.copy()
    for col in numeric_cols:
        col_min = df_num[col].min()
        col_max = df_num[col].max()
        if col_max == col_min:
            df_norm[col] = 0.5
            continue
        if '↓' in col:
            df_norm[col] = 1 - (df_num[col] - col_min) / (col_max - col_min)
        else:
            df_norm[col] = (df_num[col] - col_min) / (col_max - col_min)

    # Radar labels (cleaned)
    labels = ['CV\nAccuracy', 'CV\nF1 Macro',
              'Calibration\n(ECE inv)', 'Train\nSpeed', 'Stability']
    N = len(labels)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # close the loop

    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    colors = plt.cm.tab10.colors

    for i, (meta_name, row) in enumerate(df_norm.iterrows()):
        values = row.tolist() + row.tolist()[:1]
        ax.plot(angles, values, 'o-', lw=2, color=colors[i], label=meta_name)
        ax.fill(angles, values, alpha=0.07, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=7)
    ax.set_title(f'Meta-Learner Scorecard Radar\n{scenario_name}',
                 fontsize=12, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'../results/radar_{scenario_name.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_radar_chart(scorecard_p7c, 'P7C (7 classes, Known Consumer)')
plot_radar_chart(scorecard_p6c, 'P6C (6 classes, Known Consumer)')

---
## CELL 18 — Select the Best Meta-Learner

Selection is based on the scorecard:
1. Highest CV accuracy (primary criterion)
2. Tie-breaking: lowest ECE, then lowest std, then interpretability preference

⚠️ The winning meta-learner is selected here. The test set is **only used in Cell 19**.

In [ ]:
def select_best_meta_learner(results_dict, scenario_name):
    """
    Selects best meta-learner from CV results.
    Primary: highest CV Accuracy
    Tiebreak: lowest ECE, then lowest std
    """
    best_name  = None
    best_acc   = -1
    best_ece   = 999
    best_std   = 999

    for name, vals in results_dict.items():
        acc = vals['CV Accuracy Mean']
        ece = vals['ECE (↓ better)']
        std = vals['Stability (std↓)']

        if (acc > best_acc or
           (abs(acc - best_acc) < 0.1 and ece < best_ece) or
           (abs(acc - best_acc) < 0.1 and abs(ece - best_ece) < 0.001 and std < best_std)):
            best_name = name
            best_acc  = acc
            best_ece  = ece
            best_std  = std

    print(f'\n{"─"*55}')
    print(f'  Selected Meta-Learner — {scenario_name}')
    print(f'{"─"*55}')
    print(f'  ✓  {best_name}')
    print(f'     CV Accuracy : {best_acc:.2f}%')
    print(f'     ECE         : {best_ece:.4f}')
    print(f'     Std (stab.) : {best_std:.4f}%')
    print(f'     Trained model retrieved from results dict.')

    return best_name, results_dict[best_name]['trained_model']


best_name_p7c, best_model_p7c = select_best_meta_learner(results_p7c, 'P7C')
best_name_p6c, best_model_p6c = select_best_meta_learner(results_p6c, 'P6C')

print(f'\nFinal selections:')
print(f'  P7C best meta-learner: {best_name_p7c}')
print(f'  P6C best meta-learner: {best_name_p6c}')

---
## CELL 19 — Final Test Set Evaluation (Done Once, for Selected Models Only)

This is the only cell that uses the test set. The selected meta-learner is evaluated on test-set meta-features generated by the fully-trained base models.

In [ ]:
def final_test_evaluation(best_meta_model, trained_bases,
                           X_test, y_test, n_classes, class_names, scenario_name):
    """
    Generates test-set meta-features and evaluates the selected meta-learner.
    Called only once per scenario.
    """
    # Step 1: Generate test meta-features using trained base models
    test_meta_X = generate_test_meta_features(X_test, trained_bases, n_classes)

    # Step 2: Predict with selected meta-learner
    y_pred = best_meta_model.predict(test_meta_X)
    y_prob = best_meta_model.predict_proba(test_meta_X)

    # Step 3: Compute all metrics
    acc   = accuracy_score(y_test, y_pred)  * 100
    prec  = precision_score(y_test, y_pred, average='macro', zero_division=0) * 100
    rec   = recall_score(y_test, y_pred, average='macro', zero_division=0) * 100
    f1    = f1_score(y_test, y_pred, average='macro', zero_division=0) * 100
    ece   = compute_ece(y_test, y_prob)
    try:
        auc_score = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro') * 100
    except Exception:
        auc_score = float('nan')

    print(f'\n{"═"*55}')
    print(f'  FINAL TEST RESULTS — {scenario_name}')
    print(f'{"═"*55}')
    print(f'  Accuracy   : {acc:.2f}%')
    print(f'  Precision  : {prec:.2f}%')
    print(f'  Recall     : {rec:.2f}%')
    print(f'  F1-Score   : {f1:.2f}%')
    print(f'  AUC (OvR)  : {auc_score:.2f}%')
    print(f'  ECE        : {ece:.4f}')
    print()
    print('Per-class report:')
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    return {
        'accuracy': acc, 'precision': prec, 'recall': rec,
        'f1': f1, 'auc': auc_score, 'ece': ece,
        'y_pred': y_pred, 'y_prob': y_prob
    }


print('Running final test evaluation...')
print('(Test set is used for the first and only time below)\n')

final_p7c = final_test_evaluation(
    best_model_p7c, trained_bases_p7c,
    X_test_p7c, y_test_p7c, n_classes_p7c, class_names_p7c, 'P7C'
)

final_p6c = final_test_evaluation(
    best_model_p6c, trained_bases_p6c,
    X_test_p6c, y_test_p6c, n_classes_p6c, class_names_p6c, 'P6C'
)

---
## CELL 20 — Confusion Matrices for Final Models

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        linewidths=0.5
    )
    plt.title(f'Confusion Matrix — {title}', fontsize=12)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'../results/cm_{title.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_confusion_matrix(y_test_p7c, final_p7c['y_pred'], class_names_p7c,
                      f'Best Stacking Model ({best_name_p7c}) — P7C')

plot_confusion_matrix(y_test_p6c, final_p6c['y_pred'], class_names_p6c,
                      f'Best Stacking Model ({best_name_p6c}) — P6C')

---
## CELL 21 — ROC Curves for Final Models

In [ ]:
def plot_roc_curves(y_test, y_prob, n_classes, class_names, title):
    y_bin     = label_binarize(y_test, classes=list(range(n_classes)))
    all_fpr   = np.unique(np.concatenate(
        [roc_curve(y_bin[:, i], y_prob[:, i])[0] for i in range(n_classes)]
    ))
    mean_tpr  = np.zeros_like(all_fpr)

    plt.figure(figsize=(9, 7))
    colors = plt.cm.tab10.colors

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc_val = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=1.5, color=colors[i % 10],
                 label=f'{class_names[i]} (AUC={roc_auc_val:.2f})')
        mean_tpr += np.interp(all_fpr, fpr, tpr)

    mean_tpr  /= n_classes
    macro_auc  = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, 'k--', lw=2.5,
             label=f'Macro Average (AUC={macro_auc:.2f})')
    plt.plot([0,1],[0,1],'k:',lw=1)
    plt.xlim([0,1]); plt.ylim([0,1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curves — {title}', fontsize=12)
    plt.legend(loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'../results/roc_{title.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_roc_curves(y_test_p7c, final_p7c['y_prob'], n_classes_p7c, class_names_p7c,
                f'Best Stacking Model ({best_name_p7c}) — P7C')

plot_roc_curves(y_test_p6c, final_p6c['y_prob'], n_classes_p6c, class_names_p6c,
                f'Best Stacking Model ({best_name_p6c}) — P6C')

---
## CELL 22 — Full Thesis-Ready Summary Table

Combines scorecard + final test results into one export-ready table comparing all meta-learners and the final selected model against the paper's baseline.

In [ ]:
# Paper's reported values (from Table 1 and Table 5 of Mohammad et al., 2023)
paper_baselines = {
    'P7C': {'Soft Voting (Paper)': {'accuracy': 88.00, 'f1': 85.49, 'precision': 83.40, 'recall': 88.55}},
    'P6C': {'Soft Voting (Paper)': {'accuracy': 94.75, 'f1': 94.87, 'precision': 94.90, 'recall': 94.88}},
}

print('\n' + '='*80)
print('  FULL COMPARATIVE SUMMARY — META-LEARNER SELECTION + FINAL TEST RESULTS')
print('='*80)

for scenario, results_dict, scorecard, final_res, best_name, paper_bl in [
    ('P7C', results_p7c, scorecard_p7c, final_p7c, best_name_p7c, paper_baselines['P7C']),
    ('P6C', results_p6c, scorecard_p6c, final_p6c, best_name_p6c, paper_baselines['P6C']),
]:
    print(f'\n── {scenario} ──────────────────────────────────────────────────────────')

    rows = []
    # Paper baseline
    for model_name, vals in paper_bl.items():
        rows.append({
            'Model': model_name,
            'Source': 'Paper (reported)',
            'CV Acc (%)': '—',
            'CV Std (%)': '—',
            'ECE': '—',
            'Train Time (s)': '—',
            'Interpretable': '—',
            'Test Acc (%)': vals['accuracy'],
            'Test F1 (%)': vals['f1'],
            'Selected': ''
        })

    # All meta-learners (CV results + test results for the selected one)
    for meta_name, vals in results_dict.items():
        is_selected = (meta_name == best_name)
        rows.append({
            'Model': f'Stacking + {meta_name}',
            'Source': 'This work',
            'CV Acc (%)': round(vals['CV Accuracy Mean'], 2),
            'CV Std (%)': round(vals['Stability (std↓)'], 4),
            'ECE': round(vals['ECE (↓ better)'], 4),
            'Train Time (s)': round(vals['Train Time (s)'], 3),
            'Interpretable': vals['Interpretable'],
            'Test Acc (%)': round(final_res['accuracy'], 2) if is_selected else '(not evaluated)',
            'Test F1 (%)': round(final_res['f1'], 2) if is_selected else '(not evaluated)',
            'Selected': '★ SELECTED' if is_selected else ''
        })

    summary_df = pd.DataFrame(rows)
    display(summary_df)
    summary_df.to_csv(f'../results/full_summary_{scenario}.csv', index=False)
    print(f'  → Saved to ffull_summary_{scenario}.csv')

    # Improvement over paper
    paper_acc = list(paper_bl.values())[0]['accuracy']
    diff      = final_res['accuracy'] - paper_acc
    print(f'\n  Improvement over paper soft-voting ensemble: {diff:+.2f}%')
    print(f'  Selected meta-learner: {best_name}')
    print(f'  ECE of selected model: {final_res["ece"]:.4f} (test set)')

---
## CELL 23 — Save All Trained Models

In [ ]:
import joblib

# Save best meta-learners
joblib.dump(best_model_p7c, f'../results/best_meta_learner_p7c_{best_name_p7c.replace(" ","_")}.pkl')
joblib.dump(best_model_p6c, f'../results/best_meta_learner_p6c_{best_name_p6c.replace(" ","_")}.pkl')

# Save base models
joblib.dump(trained_bases_p7c, '../results/trained_base_models_p7c.pkl')
joblib.dump(trained_bases_p6c, '../results/trained_base_models_p6c.pkl')

# Save scalers
joblib.dump(scaler_p7c, '../results/scaler_p7c.pkl')
joblib.dump(scaler_p6c, '../results/scaler_p6c.pkl')

print('Models saved:')
print(f'  best_meta_learner_p7c_{best_name_p7c.replace(" ","_")}.pkl')
print(f'  best_meta_learner_p6c_{best_name_p6c.replace(" ","_")}.pkl')
print('  trained_base_models_p7c.pkl / p6c.pkl')
print('  scaler_p7c.pkl / scaler_p6c.pkl')

---
## Summary Table — Cell Guide

| Cell | Purpose | Uses Test Set? |
|------|---------|----------------|
| 1 | Install packages | No |
| 2 | Import all libraries + global settings | No |
| 3 | Load dataset | No |
| 4 | Preprocessing (encode, clean) | No |
| 5 | Build P7C and P6C scenario arrays | No |
| 6 | Train/test split + StandardScaler | Split only |
| 7 | Define base models (paper hyperparams) | No |
| 8 | Define 6 meta-learner candidates | No |
| 9 | Generate OOF meta-features (train only) | **No** |
| 10 | Define ECE calculator | No |
| 11 | CV comparison across all meta-learners | **No** |
| 12 | Friedman statistical test | No |
| 13 | Nemenyi post-hoc pairwise test | No |
| 14 | Calibration reliability diagrams | No |
| 15 | CV accuracy box plots | No |
| 16 | Build scorecard table (highlighted) | No |
| 17 | Radar chart across all criteria | No |
| 18 | Select best meta-learner | No |
| 19 | **Final test set evaluation** | **YES — only here** |
| 20 | Confusion matrices (final models) | Yes (via Cell 19 results) |
| 21 | ROC curves (final models) | Yes (via Cell 19 results) |
| 22 | Full comparative summary table + CSV export | Yes (via Cell 19 results) |
| 23 | Save all models to disk | No |

---

### Thesis-ready paragraph (template)

> Six meta-learner candidates were evaluated to determine the optimal combination strategy for the stacking ensemble: Logistic Regression, Ridge Classifier, Decision Tree, SVM (RBF), Gradient Boosting, and Extra Trees. Selection was performed exclusively on training data using 5-fold stratified cross-validation on out-of-fold meta-features, ensuring no information from the test set influenced model selection. Candidates were assessed across five criteria: CV accuracy, macro F1-score, Expected Calibration Error (ECE), training time, and cross-fold stability. Statistical significance of observed differences was verified using the Friedman test followed by Nemenyi post-hoc pairwise comparison. The selected meta-learner was subsequently evaluated once on the held-out test set to produce the final reported metrics.